<a href="https://colab.research.google.com/github/huseyin-karaca/s2t-tr-dev/blob/main/notebooks/colab/s2t_tr_dev_synthetic_experiment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Synthetic Regime-Switch Experiment

End-to-end notebook for the synthetic experiment in Section 4.1 of the manuscript. The flow:

1. Clone the repo and set up the `uv` environment.
2. Generate the regime-switch parquet matching `src/data/dataset.py` schema.
3. Compute training-free baselines (random, weighted random, individual base models, oracle).
4. Train the proposed hierarchical transformer router.
5. Train the MLP-pool baseline router.
6. Evaluate both checkpoints on the same test split.
7. (Optional) Mirror the logs to Google Drive.

All cells use `!uv run python -m ...` so the notebook stays reproducible and matches the CLI used elsewhere in the project.

## 1. Initial setup (clone + uv env + secrets)

In [ ]:
! git clone https://github.com/huseyin-karaca/s2t-tr-dev
%cd /content/s2t-tr-dev

from google.colab import files
files.view('/content/s2t-tr-dev')

!make create_environment
!make requirements

from google.colab import userdata
import os
os.environ['GITHUB_TOKEN'] = userdata.get('GitHubPAT')
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')

!gh auth status
!git config --global user.name 'huseyin-karaca'
!git config --global user.email 'huseyinkaraccca@gmail.com'

Cloning into 's2t-tr-dev'...
remote: Enumerating objects: 347, done.
remote: Counting objects: 100% (103/103), done.
remote: Compressing objects: 100% (72/72), done.
remote: Total 347 (delta 44), reused 83 (delta 31), pack-reused 244 (from 2)
Receiving objects: 100% (347/347), 738.71 MiB | 19.93 MiB/s, done.
Resolving deltas: 100% (130/130), done.
Updating files: 100% (139/139), done.
/content/s2t-tr-dev


<IPython.core.display.Javascript object>

uv venv --python 3.10
Using CPython 3.10.12 interpreter at: /usr/bin/python3.10
Creating virtual environment at: .venv
Activate with: source .venv/bin/activate
>>> New uv virtual environment created. Activate with:
>>> Windows: .\.venv\Scripts\activate
>>> Unix/macOS: source ./.venv/bin/activate
uv sync
Resolved 199 packages in 23ms
Prepared 195 packages in 37.05s
Installed 195 packages in 309ms
 + absl-py==2.4.0
 + aiohappyeyeballs==2.6.1
 + aiohttp==3.13.5
 + aiosignal==1.4.0
 + annotated-doc==0.0.4
 + anyio==4.13.0
 + argon2-cffi==25.1.0
 + argon2-cffi-bindings==25.1.0
 + arrow==1.4.0
 + asttokens==3.0.1
 + async-lru==2.3.0
 + async-timeout==5.0.1
 + attrs==26.1.0
 + audioread==3.1.0
 + babel==2.18.0
 + beautifulsoup4==4.14.3
 + bleach==6.3.0
 + certifi==2026.2.25
 + cffi==2.0.0
 + charset-normalizer==3.4.7
 + click==8.3.2
 + comm==0.2.3
 + contourpy==1.3.2
 + cuda-bindings==13.2.0
 + cuda-pathfinder==1.5.3
 + cuda-toolkit==13.0.2
 + cycler==0.12.1
 + datasets==3.6.0
 + debugpy==1.8

## 2. Generate the synthetic regime-switch parquet

The output file follows the same schema as the real-world parquets: three frame-level feature columns and three per-expert WER columns. With the defaults (`N=10000`, `T=128`, `R=4`, `K=3`, `--feature-dtype float16`) the file is around 5 -- 6 GB on disk; the embeddings dominate. Generation is streamed in row groups of `--write-batch-size` clips, so peak RAM is bounded by one batch (~0.7 GB at default settings) regardless of `--num-samples`. Lower `--write-batch-size` if you have very tight RAM, or set `--feature-dtype float32` if you need full-precision embeddings (doubles disk and RAM).

In [ ]:
# Smoke run first — small file to confirm the pipeline before committing to the full size.
!uv run python -m src.data.synthetic \
    --output-path data/processed/synthetic_regime_switch_smoke/combined_features.parquet \
    --num-samples 200 --frame-length 32 --seed 42

In [ ]:
# Full synthetic dataset.
!uv run python -m src.data.synthetic \
    --output-path data/processed/synthetic_regime_switch/combined_features.parquet \
    --num-samples 10000 --frame-length 128 \
    --num-regimes 4 --regime-dim 32 --noise-std 0.5 \
    --best-wer 0.15 --worst-wer 0.50 --wer-noise-std 0.02 \
    --seed 42

2026-04-26 16:14:13.430 | INFO     | src.config:<module>:9 - PROJ_ROOT path is: /content/s2t-tr-dev
2026-04-26 16:14:13,860 [INFO] __main__: Generating 10000 clips → data/processed/synthetic_regime_switch/combined_features.parquet (T=128, R=4, regime_dim=32, dims={'hubert': 1024, 'whisper': 512, 'wav2vec2': 1024}, dtype=float16)
2026-04-26 16:14:13,860 [INFO] __main__: Embedding payload: 6.55 GB total, peak per write batch: 0.33 GB (write_batch_size=500)
2026-04-26 16:15:01,769 [INFO] __main__:   wrote 2000 / 10000 clips
2026-04-26 16:15:48,024 [INFO] __main__:   wrote 4000 / 10000 clips
2026-04-26 16:16:34,273 [INFO] __main__:   wrote 6000 / 10000 clips
2026-04-26 16:17:21,414 [INFO] __main__:   wrote 8000 / 10000 clips
2026-04-26 16:18:08,756 [INFO] __main__:   wrote 10000 / 10000 clips
2026-04-26 16:18:08,758 [INFO] __main__: Sanity stats — random WER: 0.3785, oracle WER: 0.1531, per-expert mean WER: {'hubert': 0.43208104372024536, 'whisper': 0.344176709651947, 'wav2vec2': 0.3591602

## 3. Inspect the synthetic dataset structure

A four-panel sanity figure produced directly from the parquet metadata: regime centers in the latent space (PCA), the per-(r1, r2) best-expert grid, the per-expert WER table heatmaps, and a few per-clip frame trajectories. Use it to confirm that regimes are separable, the WER table is asymmetric in the ordered pair, and clips show a clean regime switch in their embeddings.

In [ ]:
!uv run python -m src.data.synthetic_inspect \
    --parquet-path data/processed/synthetic_regime_switch/combined_features.parquet \
    --output-path  reports/figures/synthetic_inspect.pdf \
    --num-trajectories 4 --seed 0

2026-04-26 16:18:09.039 | INFO     | src.config:<module>:9 - PROJ_ROOT path is: /content/s2t-tr-dev
2026-04-26 16:18:12,553 [INFO] __main__: Loaded synthetic metadata from data/processed/synthetic_regime_switch/combined_features.parquet — R=4, regime_dim=32, K=3
2026-04-26 16:18:25,251 [INFO] __main__: Wrote inspection figure to reports/figures/synthetic_inspect.pdf


## 4. Training-free baselines

These do not require a learned model: oracle, random, weighted random, and each base model on its own. The script reports numbers on the same test split that the trained routers are evaluated on (matching `--seed`, `--train-ratio`, `--val-ratio`).

In [ ]:
!uv run python -m src.training.eval_baselines \
    --parquet-path data/processed/synthetic_regime_switch/combined_features.parquet \
    --train-ratio 0.8 --val-ratio 0.1 --seed 42 --split test \
    --save-json logs/synthetic_regime_switch_baselines.json

2026-04-26 16:18:25.918 | INFO     | src.config:<module>:9 - PROJ_ROOT path is: /content/s2t-tr-dev
2026-04-26 16:18:30,585 [INFO] __main__: Loaded WER matrix from data/processed/synthetic_regime_switch/combined_features.parquet — N=10000, K=3, splits: train=8000 val=1000 test=1000
2026-04-26 16:18:30,586 [INFO] __main__: === Baselines on test split ===
2026-04-26 16:18:30,587 [INFO] __main__:   single_hubert          wer=0.4305 ± 0.0042 (n=1000)
2026-04-26 16:18:30,587 [INFO] __main__:   single_whisper         wer=0.3413 ± 0.0054 (n=1000)
2026-04-26 16:18:30,587 [INFO] __main__:   single_wav2vec2        wer=0.3618 ± 0.0052 (n=1000)
2026-04-26 16:18:30,587 [INFO] __main__:   oracle                 wer=0.1520 ± 0.0010 (n=1000)
2026-04-26 16:18:30,587 [INFO] __main__:   random                 wer=0.3803 ± 0.0051 (n=1000)
2026-04-26 16:18:30,587 [INFO] __main__:   weighted_random        wer=0.3697 ± 0.0052 (n=1000)
2026-04-26 16:18:30,587 [INFO] __main__:   weighted_random_weights: {'hube

## 5. Train the proposed hierarchical transformer router

Defaults match the manuscript: `d=256`, 2 Stage-1 layers, 1 Stage-2 layer, cross-attention bridge enabled, shared Stage-1 weights. The synthetic loss configuration uses weighted-WER plus soft cross-entropy, the best combination identified by the loss ablation (Section 4.3.1 of the manuscript).

In [ ]:
!uv run python -m src.training.train \
    --parquet-path data/processed/synthetic_regime_switch/combined_features.parquet \
    --arch hierarchical_transformer \
    --max-seq-len 256 \
    --batch-size 32 --num-workers 2 \
    --d-model 256 --n-heads 4 \
    --stage1-layers 2 --stage2-layers 1 --ffn-dim 512 \
    --dropout 0.15 \
    --learning-rate 1e-4 --weight-decay 1e-2 --warmup-steps 200 \
    --max-epochs 2 \
    --primary-weight 1.0 --aux-ce-weight 0.0 \
    --soft-ce-weight 0.5 --soft-ce-temperature 0.1 \
    --seed 42 \
    --experiment-name synthetic_regime_switch_hier

2026-04-26 16:18:31.346 | INFO     | src.config:<module>:9 - PROJ_ROOT path is: /content/s2t-tr-dev
Seed set to 42
2026-04-26 16:18:41,487 [INFO] httpx: HTTP Request: HEAD https://s3.amazonaws.com/datasets.huggingface.co/datasets/datasets/parquet/parquet.py "HTTP/1.1 404 Not Found"
Generating train split: 10000 examples [01:05, 153.84 examples/s]
2026-04-26 16:19:46,579 [INFO] src.data.dataset: Loaded 10000 samples from data/processed/synthetic_regime_switch/combined_features.parquet
2026-04-26 16:19:46,579 [INFO] __main__: Dataset splits: train=8000, val=1000, test=1000
2026-04-26 16:19:46,626 [INFO] __main__: === Model Parameter Counts ===
2026-04-26 16:19:46,626 [INFO] __main__:                  projection:     656128
2026-04-26 16:19:46,626 [INFO] __main__:                pos_encoding:          0
2026-04-26 16:19:46,626 [INFO] __main__:            model_embeddings:        768
2026-04-26 16:19:46,626 [INFO] __main__:                  cls_tokens:        768
2026-04-26 16:19:46,626 [I

## 6. Train the MLP-pool baseline router

Same loss and the same seed/splits as the proposed router; only the architecture changes. The MLP mean-pools each expert's frame sequence and feeds the concatenation through a 2-layer MLP.

In [ ]:
!uv run python -m src.training.train \
    --parquet-path data/processed/synthetic_regime_switch/combined_features.parquet \
    --arch mlp_pool \
    --max-seq-len 256 \
    --batch-size 32 --num-workers 4 \
    --mlp-hidden 1024 --mlp-layers 2 \
    --dropout 0.15 \
    --learning-rate 1e-4 --weight-decay 1e-2 --warmup-steps 200 \
    --max-epochs 2 \
    --primary-weight 1.0 --aux-ce-weight 0.0 \
    --soft-ce-weight 0.5 --soft-ce-temperature 0.1 \
    --seed 42 \
    --experiment-name synthetic_regime_switch_mlp_pool

2026-04-26 16:47:17.596 | INFO     | src.config:<module>:9 - PROJ_ROOT path is: /content/s2t-tr-dev
Seed set to 42
2026-04-26 16:47:23,960 [INFO] httpx: HTTP Request: HEAD https://s3.amazonaws.com/datasets.huggingface.co/datasets/datasets/parquet/parquet.py "HTTP/1.1 404 Not Found"
2026-04-26 16:47:23,993 [INFO] src.data.dataset: Loaded 10000 samples from data/processed/synthetic_regime_switch/combined_features.parquet
2026-04-26 16:47:23,993 [INFO] __main__: Dataset splits: train=8000, val=1000, test=1000
2026-04-26 16:47:24,040 [INFO] __main__: === Model Parameter Counts ===
2026-04-26 16:47:24,040 [INFO] __main__:                  classifier:    3675139
2026-04-26 16:47:24,040 [INFO] __main__:                       total:    3675139
Using 16bit Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enab

## 6.b Evaluate the trained checkpoints on the test split (optional, the train script already does this once)

Both runs are evaluated on the same test split (same seed and split ratios). The training script already runs a final test pass automatically and dumps results to TensorBoard; the cells below repeat it as a standalone JSON snapshot for the manuscript table.

In [ ]:
!uv run python -m src.training.evaluate \
    --checkpoint logs/synthetic_regime_switch_hier/checkpoints/last.ckpt \
    --parquet-path data/processed/synthetic_regime_switch/combined_features.parquet \
    --train-ratio 0.8 --val-ratio 0.1 --seed 42 --split test \
    --batch-size 32 --max-seq-len 256 --num-workers 4 \
    --save-json logs/synthetic_regime_switch_hier/test_results.json

2026-04-26 16:59:13.441 | INFO     | src.config:<module>:9 - PROJ_ROOT path is: /content/s2t-tr-dev
Seed set to 42
2026-04-26 16:59:18,932 [INFO] __main__: Loading checkpoint: logs/synthetic_regime_switch_hier/checkpoints/last.ckpt
2026-04-26 16:59:20,097 [INFO] httpx: HTTP Request: HEAD https://s3.amazonaws.com/datasets.huggingface.co/datasets/datasets/parquet/parquet.py "HTTP/1.1 404 Not Found"
2026-04-26 16:59:20,127 [INFO] src.data.dataset: Loaded 10000 samples from data/processed/synthetic_regime_switch/combined_features.parquet
2026-04-26 16:59:20,127 [INFO] __main__: Evaluating on split 'test' (1000 samples)
Using 16bit Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
💡 Tip: For seamless clo

In [ ]:
!uv run python -m src.training.evaluate \
    --checkpoint logs/synthetic_regime_switch_mlp_pool/checkpoints/last.ckpt \
    --parquet-path data/processed/synthetic_regime_switch/combined_features.parquet \
    --train-ratio 0.8 --val-ratio 0.1 --seed 42 --split test \
    --batch-size 32 --max-seq-len 256 --num-workers 4 \
    --save-json logs/synthetic_regime_switch_mlp_pool/test_results.json

2026-04-26 17:00:00.272 | INFO     | src.config:<module>:9 - PROJ_ROOT path is: /content/s2t-tr-dev
Seed set to 42
2026-04-26 17:00:05,732 [INFO] __main__: Loading checkpoint: logs/synthetic_regime_switch_mlp_pool/checkpoints/last.ckpt
2026-04-26 17:00:06,882 [INFO] httpx: HTTP Request: HEAD https://s3.amazonaws.com/datasets.huggingface.co/datasets/datasets/parquet/parquet.py "HTTP/1.1 404 Not Found"
2026-04-26 17:00:06,913 [INFO] src.data.dataset: Loaded 10000 samples from data/processed/synthetic_regime_switch/combined_features.parquet
2026-04-26 17:00:06,914 [INFO] __main__: Evaluating on split 'test' (1000 samples)
Using 16bit Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
💡 Tip: For seamless

## 7. Post-hoc visualizations of both trained checkpoints

Confusion matrices, per-clip WER scatters, per-base-model selection frequencies, and overlaid training curves. This subcommand is generic — the same calls are reused for the real-world AMI / VoxPopuli runs later. All figures are written under `reports/figures/synthetic_regime_switch/`.

In [ ]:
!uv run python -m src.training.visualize predictions \
    --parquet-path data/processed/synthetic_regime_switch/combined_features.parquet \
    --ckpt proposed=logs/synthetic_regime_switch_hier/checkpoints/last.ckpt \
    --ckpt mlp_pool=logs/synthetic_regime_switch_mlp_pool/checkpoints/last.ckpt \
    --output-dir reports/figures/synthetic_regime_switch \
    --split test --train-ratio 0.8 --val-ratio 0.1 --seed 42 \
    --max-seq-len 256 --batch-size 32 --num-workers 4

2026-04-26 17:00:46.710 | INFO     | src.config:<module>:9 - PROJ_ROOT path is: /content/s2t-tr-dev
2026-04-26 17:00:53,050 [INFO] httpx: HTTP Request: HEAD https://s3.amazonaws.com/datasets.huggingface.co/datasets/datasets/parquet/parquet.py "HTTP/1.1 404 Not Found"
2026-04-26 17:00:53,083 [INFO] src.data.dataset: Loaded 10000 samples from data/processed/synthetic_regime_switch/combined_features.parquet
2026-04-26 17:00:53,106 [INFO] __main__: Device: cuda; split=test (n=1000)
2026-04-26 17:00:53,106 [INFO] __main__: === checkpoint proposed ← logs/synthetic_regime_switch_hier/checkpoints/last.ckpt ===
/content/s2t-tr-dev/.venv/lib/python3.10/site-packages/torch/nn/modules/transformer.py:531: UserWarning: The PyTorch API of nested tensors is in prototype stage and will change in the near future. We recommend specifying layout=torch.jagged when constructing a nested tensor, as this layout receives active development, has better operator coverage, and works with torch.compile. (Triggered

In [ ]:
!uv run python -m src.training.visualize curves \
    --logdir proposed=logs/synthetic_regime_switch_hier \
    --logdir mlp_pool=logs/synthetic_regime_switch_mlp_pool \
    --output-path reports/figures/synthetic_regime_switch/curves.pdf

2026-04-26 17:02:12.332 | INFO     | src.config:<module>:9 - PROJ_ROOT path is: /content/s2t-tr-dev
2026-04-26 17:02:17,992 [INFO] __main__:   proposed :: val/selected_wer — 2 points
2026-04-26 17:02:17,993 [INFO] __main__:   proposed :: val/oracle_wer — 2 points
2026-04-26 17:02:17,993 [INFO] __main__:   proposed :: val/selection_accuracy — 2 points
2026-04-26 17:02:18,021 [INFO] __main__:   mlp_pool :: val/selected_wer — 2 points
2026-04-26 17:02:18,021 [INFO] __main__:   mlp_pool :: val/oracle_wer — 2 points
2026-04-26 17:02:18,021 [INFO] __main__:   mlp_pool :: val/selection_accuracy — 2 points
2026-04-26 17:02:19,326 [INFO] __main__: Wrote curves figure to reports/figures/synthetic_regime_switch/curves.pdf


## 8. Sweep over the number of regimes (manuscript Fig.~\ref{fig:synthetic_sweep})

Re-runs the entire pipeline (data generation → train both architectures → checkpoint evaluation → training-free baselines) for each value of `R`, then renders the manuscript sweep figure from the resulting JSON. The sweep uses smaller-than-default settings (`--num-samples 5000`, `--max-epochs 30`) so the full sweep stays within a typical Colab session; bump them up if you have time and want tighter curves. Per-R parquets are deleted as soon as their cell finishes (pass `--keep-parquets` to retain them).

In [ ]:
!uv run python -m src.experiments.synthetic_sweep run \
    --output-dir reports/sweeps/synthetic_R \
    --r-values 2,4,6,8 \
    --num-samples 5000 --frame-length 128 --regime-dim 32 --noise-std 0.5 \
    --max-seq-len 256 --batch-size 64 --num-workers 4 \
    --max-epochs 1 \
    --primary-weight 1.0 --aux-ce-weight 0.0 \
    --soft-ce-weight 0.5 --soft-ce-temperature 0.1 \
    --seed 42 --keep-parquets

2026-04-26 17:02:20.567 | INFO     | src.config:<module>:9 - PROJ_ROOT path is: /content/s2t-tr-dev
2026-04-26 17:02:21,087 [INFO] __main__: Sweep R values: [2, 4, 6, 8] — output dir: reports/sweeps/synthetic_R
2026-04-26 17:02:21,088 [INFO] __main__: >>> generate synthetic parquet (R=2)
    /content/s2t-tr-dev/.venv/bin/python3 -m src.data.synthetic --output-path reports/sweeps/synthetic_R/R2/combined_features.parquet --num-samples 5000 --frame-length 128 --num-regimes 2 --regime-dim 32 --noise-std 0.5 --feature-dtype float16 --write-batch-size 500 --seed 42
2026-04-26 17:02:21.182 | INFO     | src.config:<module>:9 - PROJ_ROOT path is: /content/s2t-tr-dev
2026-04-26 17:02:21,352 [INFO] __main__: Generating 5000 clips → reports/sweeps/synthetic_R/R2/combined_features.parquet (T=128, R=2, regime_dim=32, dims={'hubert': 1024, 'whisper': 512, 'wav2vec2': 1024}, dtype=float16)
2026-04-26 17:02:21,352 [INFO] __main__: Embedding payload: 3.28 GB total, peak per write batch: 0.33 GB (write_b

In [ ]:
!uv run python -m src.experiments.synthetic_sweep figure \
    --results reports/sweeps/synthetic_R/sweep_results.json \
    --output-path reports/manuscript/figures/synthetic_sweep.pdf

2026-04-26 17:45:27.032 | INFO     | src.config:<module>:9 - PROJ_ROOT path is: /content/s2t-tr-dev
2026-04-26 17:45:28,371 [INFO] __main__: Wrote sweep figure to reports/manuscript/figures/synthetic_sweep.pdf


In [ ]:
from IPython.display import IFrame
import os

pdf_path = "reports/manuscript/figures/synthetic_sweep.pdf"

if os.path.exists(pdf_path):
    display(IFrame(pdf_path, width=800, height=600))
else:
    print(f"Hata: {pdf_path} bulunamadı. Lütfen bir önceki hücrenin başarıyla çalıştığından emin olun.")

## 9. (Optional) Mirror logs and result JSONs to Google Drive

Useful for keeping a copy of the run artefacts after the Colab session ends.

In [ ]:
from google.colab import drive
drive.mount('/gdrive')

Mounted at /gdrive


In [ ]:
!mkdir -p "/gdrive/My Drive/s2t-tr-dev/logs"
!cp -r logs/synthetic_regime_switch_hier      "/gdrive/My Drive/s2t-tr-dev/logs/"
!cp -r logs/synthetic_regime_switch_mlp_pool   "/gdrive/My Drive/s2t-tr-dev/logs/"
!cp     logs/synthetic_regime_switch_baselines.json "/gdrive/My Drive/s2t-tr-dev/logs/"

In [ ]:
! git add .
! git commit -m "synthetic experiment zero to end run, next analyze what has happened"
! git push origin main

[main 8baf82d] synthetic experiment zero to end run, next analyze what has happened
 100 files changed, 1341 insertions(+)
 create mode 100644 logs/synthetic_regime_switch_baselines.json
 create mode 100644 logs/synthetic_regime_switch_hier/checkpoints/best-epoch=00.ckpt
 create mode 100644 logs/synthetic_regime_switch_hier/checkpoints/best-epoch=01.ckpt
 create mode 100644 logs/synthetic_regime_switch_hier/checkpoints/last.ckpt
 create mode 100644 logs/synthetic_regime_switch_hier/config.json
 create mode 100644 logs/synthetic_regime_switch_hier/test_results.json
 create mode 100644 logs/synthetic_regime_switch_hier/version_0/events.out.tfevents.1777220387.b181b312fb1c.11459.0
 create mode 100644 logs/synthetic_regime_switch_hier/version_0/events.out.tfevents.1777221876.b181b312fb1c.11459.1
 create mode 100644 logs/synthetic_regime_switch_hier/version_0/hparams.yaml
 create mode 100644 logs/synthetic_regime_switch_mlp_pool/checkpoints/best-epoch=00.ckpt
 create mode 100644 logs/synthe

## 10. (Optional) Release the runtime

In [ ]:
from google.colab import runtime
runtime.unassign()